In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2 as cv
import scipy.signal

# Fourier frequencies of an image

In [ ]:
!wget -nc https://www.dais.unive.it/~bergamasco/teachingfiles/cvstuff/apollo.jpg

In [ ]:
I = cv.imread("apollo.jpg", cv.IMREAD_GRAYSCALE )
S = np.fft.fft2(I, norm="ortho")

S_shift = np.fft.fftshift(S)
M,N = S_shift.shape



plt.figure( figsize=(20,5) )
plt.subplot(1,4,1)
plt.imshow(I, cmap="gray")
plt.subplot(1,4,2)
plt.imshow( np.log(np.abs(S)+1), interpolation="none")
plt.title("Power spectrum")
plt.subplot(1,4,3)
plt.imshow( np.log(np.abs( S_shift )+1), interpolation="none")
plt.title("Power spectrum (shifted)")
plt.subplot(1,4,4)
plt.imshow( np.angle( S_shift ), interpolation="none", cmap="hsv")
plt.title("Phase angle (shifted)")


In [ ]:
from ipywidgets import interact, widgets

Sidx = np.argsort( np.abs(S.flatten()) )

def keep_frequencies( nfreqs ):
    S2 = np.copy(S)
    S2[ np.unravel_index(Sidx[:-nfreqs],S2.shape) ]=0

    I2 = np.real( np.fft.ifft2( S2, norm="ortho" ) )

    plt.figure( figsize=(15,5) )
    plt.subplot(1,3,1)
    plt.imshow(I, cmap="gray", vmin=0, vmax=255)
    plt.subplot(1,3,2)
    plt.imshow(I2, cmap="gray", vmin=0, vmax=255)
    plt.subplot(1,3,3)
    plt.imshow( np.log(np.abs(np.fft.fftshift(S2))+1), interpolation="none")

interact( keep_frequencies, nfreqs=(1,10000,2) )

# Ideal lowpass filter

In [ ]:
from ipywidgets import interact, widgets


def lowpass( cutoff ):

    S2 = np.copy(S)

    for u in range(S2.shape[1]//2+1):
        for v in range(S2.shape[0]//2+1):
            if np.sqrt( u**2 + v**2 )>cutoff:
                S2[v,u] = 0
                S2[-v,u] = 0
                S2[v,-u] = 0
                S2[-v,-u] = 0

    I2 = np.real( np.fft.ifft2( S2, norm="ortho" ) )

    plt.figure( figsize=(20,5) )
    plt.subplot(1,3,1)
    plt.imshow(I, cmap="gray", vmin=0, vmax=255)
    plt.subplot(1,3,2)
    plt.imshow(I2, cmap="gray", vmin=0, vmax=255)
    plt.subplot(1,3,3)
    plt.imshow( np.log(np.abs(np.fft.fftshift(S2))+1), interpolation="none")

interact( lowpass, cutoff=(1,100,2) )

# Gaussian low-pass filter

In [ ]:
w = np.expand_dims( scipy.signal.windows.gaussian( 25, 5.0 ), axis=-1)
f, [ax0, ax1] = plt.subplots(1,2, figsize=(10,5))

ax0.plot(w)
ax0.set_title("1D Gaussian window")

W = w @ w.T
ax1.imshow(W)
ax1.set_title("2D Gaussian window")

In [ ]:
Wf = np.fft.fftshift( np.fft.fft2( W, S.shape ) )


f, [ax0, ax1] = plt.subplots(1,2, figsize=(10,10) )
ax0.imshow(W, interpolation=None)
ax0.set_title("Filter (spatial)")

ax1.imshow( np.abs(Wf), interpolation=None)
ax1.set_title("Filter (frequency)")


I1_filtered = np.real( np.fft.ifft2( np.fft.ifftshift(S_shift*Wf)) )
plt.figure()
plt.imshow( I1_filtered, cmap="gray")

# Ideal high-pass filter

In [ ]:
from ipywidgets import interact, widgets


def lowpass( cutoff ):

    S2 = np.copy(S)

    for u in range(S2.shape[1]//2+1):
        for v in range(S2.shape[0]//2+1):
            if np.sqrt( u**2 + v**2 )<cutoff:
                S2[v,u] = 0
                S2[-v,u] = 0
                S2[v,-u] = 0
                S2[-v,-u] = 0

    I2 = np.real( np.fft.ifft2( S2, norm="ortho" ) )

    plt.figure( figsize=(20,5) )
    plt.subplot(1,3,1)
    plt.imshow(I, cmap="gray", vmin=0, vmax=255)
    plt.subplot(1,3,2)
    plt.imshow(I2, cmap="gray", vmin=0, vmax=255)
    plt.subplot(1,3,3)
    plt.imshow( np.log(np.abs(np.fft.fftshift(S2))+1), interpolation="none")

interact( lowpass, cutoff=(1,100,2) )